# **Konfiguracja Środowiska**

W tej sekcji:
- Sprawdzamy dostępność GPU (procesora graficznego)
- Montujemy Google Drive do dostępu do danych
- Importujemy wszystkie niezbędne biblioteki

In [ ]:
# Wymuszenie użycia TensorFlow 2.x (nowsza wersja)
%tensorflow_version 2.x
import tensorflow as tf

# Sprawdzenie czy GPU jest dostępne
# GPU (Graphics Processing Unit) przyspiesza trening nawet 10-100x!
# W Google Colab wybierz: Runtime -> Change runtime type -> GPU
device_name = tf.test.gpu_device_name()
if device_name != '/device:GPU:0':
    raise SystemError('GPU device not found')
print('Znaleziono GPU: {}'.format(device_name))

In [ ]:
# Montowanie Google Drive
# Pozwala to na dostęp do plików zapisanych w Twoim Google Drive
# Po uruchomieniu kliknij link i autoryzuj dostęp
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
# Import wszystkich niezbędnych bibliotek

# Biblioteki do pracy z danymi
import numpy as np                              # Operacje na tablicach numerycznych
import pickle                                   # Zapisywanie i wczytywanie danych
import matplotlib.pyplot as plt                 # Rysowanie wykresów
from sklearn.model_selection import train_test_split  # Podział danych
from imblearn.over_sampling import SMOTE        # Upsampling (nadpróbkowanie)
import collections                              # Liczenie wystąpień

# Biblioteki TensorFlow/Keras do budowy sieci neuronowej
from tensorflow.keras import Model
from tensorflow.keras.models import Sequential  # Model sekwencyjny (warstwa po warstwie)
from tensorflow.keras.layers.experimental.preprocessing import Rescaling  # Skalowanie danych
from tensorflow.keras.layers import Conv2D, MaxPool2D, Dense, Dropout, Flatten, Activation
from tensorflow.keras.layers import BatchNormalization, GlobalAveragePooling2D
from tensorflow.keras.losses import categorical_crossentropy  # Funkcja straty
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.preprocessing.image import ImageDataGenerator  # Augmentacja danych
from tensorflow.keras.optimizers import Adam    # Optymalizator Adam
from tensorflow.keras.initializers import HeNormal  # Inicjalizacja wag

# **Wczytywanie Danych**

Wczytujemy przygotowane wcześniej dane z pliku pickle.
Dane zawierają obrazy twarzy (48x48 pikseli) i odpowiadające im etykiety emocji.

In [ ]:
# Kopiowanie folderu projektu z Google Drive do katalogu roboczego
# UWAGA: Dostosuj ścieżkę do swojej struktury folderów!
%cp -av "/content/gdrive/MyDrive/loopQ/project" "/content"

# Przejście do katalogu projektu
%cd /content/project/

In [ ]:
def load_data(data_file):
    """
    Wczytuje dane z pliku pickle.
    
    Parametry:
    ----------
    data_file : str
        Ścieżka do pliku .p (pickle) z danymi
    
    Zwraca:
    -------
    x_data : numpy array
        Obrazy (shape: [n_samples, height, width])
    y_data : numpy array  
        Etykiety emocji (shape: [n_samples])
    """
    print('Wczytywanie danych...')
    # Otwarcie pliku w trybie odczytu binarnego
    with open(data_file, 'rb') as f:
        # Deserializacja danych z pickle
        pickle_data = pickle.load(f)
        x_data = pickle_data['x_data']  # Obrazy
        y_data = pickle_data['y_data']  # Etykiety
    print('Dane wczytane.')
    return x_data, y_data

In [ ]:
# Ścieżka do pliku z danymi treningowymi
data_file = 'data/train_data.p'

# Wczytanie danych
images, labels = load_data(data_file)

# Wyświetlenie podstawowych informacji o danych
n_samples = labels.shape[0]
print('Całkowita liczba próbek:', n_samples)
print('Kształt tablicy obrazów:', images.shape)  # [n_samples, 48, 48]
print('Kształt tablicy etykiet:', labels.shape)  # [n_samples]

# **Eksploracja Danych**

Przed treningiem warto zbadać dane:
- Jakie emocje występują w zbiorze?
- Czy dane są zbalansowane (równa liczba przykładów każdej emocji)?
- Jak wyglądają przykładowe obrazy?

In [ ]:
# Słownik mapujący numery emocji na ich nazwy
# W zbiorze danych każda emocja jest reprezentowana przez liczbę 0-6
emotions = {
    0: 'Angry',      # Złość
    1: 'Disgust',    # Obrzydzenie
    2: 'Fear',       # Strach
    3: 'Happy',      # Radość
    4: 'Sad',        # Smutek
    5: 'Surprise',   # Zaskoczenie
    6: 'Neutral'     # Neutralność
}

# Liczba klas (kategorii) do klasyfikacji
num_classes = len(emotions)

In [ ]:
def plot_sample_distribution(labels):
    """
    Rysuje wykres słupkowy pokazujący rozkład próbek dla każdej emocji.
    
    To pomaga zidentyfikować niezbalansowane dane - sytuację gdy niektóre
    emocje występują znacznie częściej niż inne.
    """
    # Zliczenie unikalnych wartości i ich wystąpień
    classes, cnts = np.unique(labels, return_counts=True)
    
    # Tworzenie wykresu
    plt.figure(figsize=(12, 5))
    plt.barh(list(emotions.values()), cnts, height=0.6)
    
    # Dodanie etykiet z liczbami na słupkach
    for i, v in enumerate(cnts):
        plt.text(v, i, ' '+str(v), va='center')
    
    plt.xlabel('Liczba próbek')
    plt.title("Rozkład próbek w zbiorze danych")

# Wyświetlenie rozkładu
plot_sample_distribution(labels)

In [ ]:
def show_images(images, labels, col=5):
    """
    Wyświetla siatkę obrazów z ich etykietami.
    
    Parametry:
    ----------
    images : numpy array
        Tablica obrazów do wyświetlenia
    labels : numpy array
        Odpowiadające etykiety
    col : int
        Liczba kolumn w siatce
    """
    n = images.shape[0]
    row = np.ceil(n / col)  # Obliczenie liczby wierszy
    
    fig = plt.figure(figsize=(2*col, 2*row))
    for i in range(n):
        fig.add_subplot(row, col, i+1)
        # Wyświetlenie obrazu w skali szarości
        plt.imshow(images[i], cmap='gray')
        # Tytuł = nazwa emocji
        plt.title(emotions[labels[i]])
        # Ukrycie osi (estetyka)
        plt.xticks([]), plt.yticks([])
    plt.show()

# Wyświetlenie pierwszych 25 obrazów
show_images(images[:25], labels[:25])

In [ ]:
def show_one_emotion(images, labels, id, start=0, num=25):
    """
    Wyświetla obrazy tylko jednej wybranej emocji.
    
    Przydatne do zbadania jak wyglądają przykłady konkretnej emocji.
    """
    # Filtrowanie: wybierz tylko obrazy z daną emocją
    image_x = images[labels==id]
    label_x = labels[labels==id]
    
    # Wyświetlenie wybranych obrazów
    show_images(image_x[start:start+num], label_x[start:start+num])

# Przykład: pokaż 25 przykładów emocji 'Disgust' (id=1)
show_one_emotion(images, labels, id=1)

# **Podział Zbioru Danych**

Dzielimy dane na 3 zbiory:
- **Treningowy (60%)** - do uczenia modelu
- **Walidacyjny (20%)** - do monitorowania postępów podczas treningu
- **Testowy (20%)** - do końcowej ewaluacji modelu

**Dlaczego to ważne?**
- Zbiór testowy NIE jest używany podczas treningu
- Pozwala to sprawdzić czy model generalizuje (działa na nowych danych)

In [ ]:
# Pierwszy podział: 80% train+val, 20% test
# random_state=42 zapewnia powtarzalność (zawsze ten sam podział)
image_train, image_test, label_train, label_test = train_test_split(
    images, labels, test_size=0.2, random_state=42)

# Drugi podział: z 80% bierzemy 75% train (60% całości) i 25% val (20% całości)
image_train, image_val, label_train, label_val = train_test_split(
    image_train, label_train, test_size=0.2, random_state=42)

# Wyświetlenie rozmiarów zbiorów
print('Próbki treningowe:', label_train.shape[0])
print('Próbki walidacyjne:', label_val.shape[0])
print('Próbki testowe:', label_test.shape[0])

# **Upsampling Danych Treningowych**

**Problem:** Niektóre emocje (np. Disgust) występują rzadziej niż inne (np. Happy).

**Rozwiązanie:** SMOTE (Synthetic Minority Over-sampling Technique)
- Tworzy syntetyczne (sztuczne) przykłady rzadkich klas
- Interpoluje między istniejącymi przykładami
- Balansuje zbiór danych

**Efekt:** Model lepiej uczy się rozpoznawać wszystkie emocje!

In [ ]:
def upsampling(x, y, strategy):
    """
    Wykonuje upsampling przy użyciu SMOTE.
    
    Parametry:
    ----------
    x : numpy array
        Obrazy (shape: [n_samples, height, width])
    y : numpy array
        Etykiety
    strategy : str lub dict
        'auto' - automatyczne zbalansowanie wszystkich klas
        dict - precyzyjna kontrola liczby próbek dla każdej klasy
    
    Zwraca:
    -------
    x_up, y_up : upsampled dane
    """
    (n, w, h) = x.shape
    
    # Utworzenie obiektu SMOTE
    sm = SMOTE(sampling_strategy=strategy, random_state=42)
    
    # SMOTE wymaga danych 2D, więc spłaszczamy obrazy
    # [n, 48, 48] -> [n, 48*48=2304]
    x_flat = x.reshape((n,-1))
    
    # Wykonanie upsampleing
    x_up, y_up = sm.fit_resample(x_flat, y)
    
    # Przywrócenie oryginalnego kształtu obrazów
    n_up = x_up.shape[0]
    x_up = x_up.reshape((n_up,w,h))

    return x_up, y_up

In [ ]:
# Wyświetlenie rozkładu PRZED upsamplingiem
# collections.Counter() zlicza wystąpienia każdej wartości
print("Rozkład etykiet PRZED upsamplingiem:")
collections.Counter(label_train)

In [ ]:
# Wykonanie upsampingu ze strategią 'auto'
# 'auto' oznacza: wyrównaj wszystkie klasy do liczności najliczniejszej klasy
image_train_up, label_train_up = upsampling(image_train, label_train, 'auto')

In [ ]:
# Wyświetlenie rozkładu PO upsamplingowaniu
# Wszystkie klasy powinny mieć teraz podobną liczbę próbek
print("Rozkład etykiet PO upsamplingowaniu:")
collections.Counter(label_train_up)

In [ ]:
# Wyświetlenie syntetycznych (wygenerowanych przez SMOTE) przykładów
# Indeksy 4000+ to na pewno syntetyczne obrazy (bo oryginalnych było mniej)
for i in range(num_classes):
    if i == 3:  # Pomijamy Happy (była najliczniejsza, nie ma nowych przykładów)
        continue
    print(f"Syntetyczne przykłady emocji: {emotions[i]}")
    show_one_emotion(image_train_up, label_train_up, id=i, start=4000, num=5)

# **Funkcje Pomocnicze**

Zestaw funkcji do:
- Przetwarzania danych (one-hot encoding, normalizacja)
- Wizualizacji wyników treningu
- Ewaluacji modelu

In [ ]:
def one_hot_encoding(labels, num_classes):
    """
    Konwertuje etykiety do formatu one-hot.
    
    Przykład:
    3 -> [0, 0, 0, 1, 0, 0, 0]  (dla 7 klas)
    
    Dlaczego?
    Sieci neuronowe lepiej uczą się z reprezentacją one-hot
    niż z pojedynczymi liczbami.
    """
    return tf.keras.utils.to_categorical(labels, num_classes)

In [ ]:
def reshape_images(images, channel=1, resize=None):
    """
    Przygotowuje obrazy do sieci neuronowej.
    
    Parametry:
    ----------
    images : numpy array
        Obrazy [n, h, w]
    channel : int
        Liczba kanałów (1=grayscale, 3=RGB)
    resize : tuple lub None
        Nowy rozmiar (h, w) jeśli potrzebna zmiana rozmiaru
    
    Zwraca:
    -------
    x : tensor [n, h, w, c]
    """
    # Dodanie wymiaru kanału: [n, h, w] -> [n, h, w, 1]
    x = tf.expand_dims(tf.convert_to_tensor(images), axis=3)
    
    # Jeśli potrzeba więcej kanałów (np. dla modeli pre-trained na RGB)
    if channel > 1:
        x = tf.repeat(x, channel, axis=3)
    
    # Zmiana rozmiaru jeśli podano
    if resize is not None:
        x = tf.image.resize(x, resize)
    
    return x

In [ ]:
def pre_processing(images, labels, num_classes, channel=1, resize=None, one_hot=True):
    """
    Kompleksowe przetwarzanie danych wejściowych.
    
    Łączy reshape_images i one_hot_encoding w jedną funkcję.
    """
    x = reshape_images(images, channel, resize)
    y = one_hot_encoding(labels, num_classes)
    return x, y

In [ ]:
def plot_metrics(history):
    """
    Rysuje wykresy loss i accuracy podczas treningu.
    
    Parametry:
    ----------
    history : Keras History object
        Zwracany przez model.fit()
    
    Wykresy pokazują:
    - Loss (strata) - powinien maleć
    - Accuracy (dokładność) - powinna rosnąć
    - Training vs Validation - czy model overfittuje?
    """
    metrics = ['loss', 'accuracy']
    plt.figure(figsize=(15, 6))
    plt.rc('font', size=12)
    
    for n, metric in enumerate(metrics):
        name = metric.capitalize()
        plt.subplot(1,2,n+1)
        
        # Wykres treningu
        plt.plot(history.epoch, history.history[metric], 
                 label='Trening', lw=3, color='navy')
        
        # Wykres walidacji
        plt.plot(history.epoch, history.history['val_'+metric], 
                 lw=3, label='Walidacja', color='deeppink')
        
        plt.xlabel('Epoka')
        plt.ylabel(name)
        plt.title(name + ' modelu')
        plt.legend()
    
    plt.show()

In [ ]:
def evaluate_class(model, x_test, y_test):
    """
    Ewaluuje model osobno dla każdej klasy emocji.
    
    Pokazuje które emocje model rozpoznaje najlepiej,
    a które sprawiają mu trudność.
    """
    # Konwersja one-hot z powrotem do indeksów
    labels = np.argmax(y_test, axis=1)
    
    print('{:<15}Dokładność'.format('Emocja'))
    print('-'*23)
    
    # Ewaluacja dla każdej emocji osobno
    for i in range(num_classes):
        # Wybierz tylko przykłady danej emocji
        x = x_test[labels==i]
        y = y_test[labels==i]
        
        # Oblicz dokładność
        loss, acc = model.evaluate(x, y, verbose=0)
        print('{:<15}{:.1%}'.format(emotions[i], acc))
    
    print('-'*23)
    
    # Ogólna dokładność
    loss, acc = model.evaluate(x_test, y_test, verbose=0)
    print('{:<15}{:.1%}'.format('Ogółem', acc))

# **Budowa i Trening Modelu**

W tej sekcji:
1. Definiujemy architekturę sieci VGGNet
2. Przygotowujemy dane do treningu (augmentacja)
3. Trenujemy model
4. Wizualizujemy wyniki

In [ ]:
def model_checkpoint_cb(file_path):
    """
    Tworzy callback do zapisywania najlepszego modelu.
    
    Podczas treningu model jest zapisywany tylko gdy
    val_accuracy się poprawia. Dzięki temu mamy zawsze
    najlepszą wersję modelu.
    """
    return ModelCheckpoint(
        file_path, 
        monitor='val_accuracy',  # Monitoruj dokładność walidacyjną
        mode='max',               # Maksymalizuj (większa accuracy = lepiej)
        save_best_only=True,      # Zapisz tylko najlepszy model
        save_weights_only=True)   # Zapisz tylko wagi (nie całą architekturę)

In [ ]:
# Przetwarzanie danych do formatu wymaganego przez sieć
x_train, y_train = pre_processing(image_train_up, label_train_up, num_classes)
x_val, y_val = pre_processing(image_val, label_val, num_classes)
x_test, y_test = pre_processing(image_test, label_test, num_classes)

# AUGMENTACJA DANYCH TRENINGOWYCH
# ImageDataGenerator tworzy losowe wariacje obrazów "w locie" podczas treningu
# Dzięki temu model widzi więcej różnorodnych przykładów
train_datagen = ImageDataGenerator(
    rotation_range=30,        # Losowa rotacja +/- 30 stopni
    shear_range=0.2,          # Losowe przesunięcie
    width_shift_range=0.2,    # Przesunięcie w poziomie
    height_shift_range=0.2,   # Przesunięcie w pionie
    zoom_range=0.1,           # Losowe przybliżenie/oddalenie
    horizontal_flip=True)     # Losowe odbicie poziome

# Dla danych walidacyjnych NIE stosujemy augmentacji
# Chcemy sprawdzić jak model radzi sobie na "normalnych" danych
val_datagen = ImageDataGenerator()

# Rozmiar batcha - ile obrazów przetwarzamy jednocześnie
# 128 to dobry kompromis między szybkością a pamięcią GPU
batch_size = 128

# Generatory danych
train_generator = train_datagen.flow(x_train, y_train, batch_size=batch_size)
val_generator = val_datagen.flow(x_val, y_val)

# Liczba kroków w jednej epoce
steps_per_epoch = train_generator.n // train_generator.batch_size

# Kształt pojedynczego obrazu wejściowego
input_shape = x_train[0].shape  # (48, 48, 1)

In [ ]:
class VGGNet(Sequential):
    """
    Architektura VGGNet dla rozpoznawania emocji.
    
    Struktura:
    - 8 warstw konwolucyjnych (Conv2D) - wykrywanie cech
    - 4 warstwy pooling (MaxPool2D) - redukcja rozmiaru
    - 3 warstwy fully connected (Dense) - klasyfikacja
    - Regularyzacja: Dropout, BatchNormalization
    
    Parametry:
    ----------
    input_shape : tuple
        Kształt obrazu wejściowego (48, 48, 1)
    num_classes : int
        Liczba klas do klasyfikacji (7)
    checkpoint_path : str
        Ścieżka do zapisu wag modelu
    lr : float
        Learning rate (szybkość uczenia)
    """
    def __init__(self, input_shape, num_classes, checkpoint_path, lr=1e-3):
        super().__init__()
        
        # WARSTWA 0: Normalizacja pikseli z [0, 255] do [0, 1]
        self.add(Rescaling(1./255, input_shape=input_shape))
        
        # BLOK 1: 2x Conv + BatchNorm + MaxPool + Dropout
        # 64 filtry 3x3, aktywacja ReLU
        self.add(Conv2D(64, (3, 3), activation='relu', kernel_initializer='he_normal'))
        self.add(BatchNormalization())  # Stabilizacja uczenia
        self.add(Conv2D(64, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same'))
        self.add(BatchNormalization())
        self.add(MaxPool2D())  # Zmniejszenie rozmiaru 2x
        self.add(Dropout(0.5))  # Wyłącz losowo 50% neuronów (zapobiega overfitting)

        # BLOK 2: 2x Conv + BatchNorm + MaxPool + Dropout
        # 128 filtrów - więcej niż w poprzednim bloku
        self.add(Conv2D(128, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same'))
        self.add(BatchNormalization())
        self.add(Conv2D(128, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same'))
        self.add(BatchNormalization())
        self.add(MaxPool2D())
        self.add(Dropout(0.4))

        # BLOK 3: 2x Conv + BatchNorm + MaxPool + Dropout
        # 256 filtrów - jeszcze więcej cech
        self.add(Conv2D(256, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same'))
        self.add(BatchNormalization())
        self.add(Conv2D(256, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same'))
        self.add(BatchNormalization())
        self.add(MaxPool2D())
        self.add(Dropout(0.5))

        # BLOK 4: 2x Conv + BatchNorm + MaxPool + Dropout
        # 512 filtrów - najbardziej abstrakcyjne cechy
        self.add(Conv2D(512, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same'))
        self.add(BatchNormalization())
        self.add(Conv2D(512, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same'))
        self.add(BatchNormalization())
        self.add(MaxPool2D())
        self.add(Dropout(0.4))

        # FLATTEN: Spłaszczenie do 1D [batch, features]
        self.add(Flatten())
        
        # WARSTWY FULLY CONNECTED (Klasyfikacja)
        self.add(Dense(1024, activation='relu'))  # 1024 neurony
        self.add(Dropout(0.5))
        self.add(Dense(256, activation='relu'))   # 256 neuronów

        # WARSTWA WYJŚCIOWA: num_classes neuronów z softmax
        # Softmax konwertuje do prawdopodobieństw (suma = 1)
        self.add(Dense(num_classes, activation='softmax'))

        # KOMPILACJA MODELU
        self.compile(
            optimizer=Adam(learning_rate=lr),      # Optymalizator Adam
            loss=categorical_crossentropy,         # Funkcja straty dla multi-class
            metrics=['accuracy'])                  # Metryka: dokładność
        
        self.checkpoint_path = checkpoint_path

In [ ]:
# Utworzenie instancji modelu
model = VGGNet(input_shape, num_classes, 'run/vggnet_up.h5')

# Wyświetlenie podsumowania architektury
# Pokazuje: warstwy, kształty, liczby parametrów
model.summary()

In [ ]:
# KONFIGURACJA TRENINGU

# Liczba epok (pełnych przebiegów przez dane)
# Model zobaczy każdy przykład treningowy 200 razy (lub mniej jeśli early stopping)
epochs = 200

# Callback 1: Model Checkpoint - zapisz najlepszy model
cp = model_checkpoint_cb(model.checkpoint_path)

# Callback 2: Learning Rate Reduction - zmniejsz LR gdy model przestaje się poprawiać
# Jeśli val_loss nie maleje przez 5 epok, zmniejsz LR o połowę
lr = ReduceLROnPlateau(
    monitor='val_loss',  # Monitoruj stratę walidacyjną
    factor=0.5,          # Zmniejsz LR o połowę
    patience=5,          # Czekaj 5 epok
    verbose=1,           # Wyświetl komunikaty
    min_lr=1e-10)        # Minimalna wartość LR

# Callback 3: Early Stopping - zatrzymaj trening gdy model przestaje się poprawiać
# Jeśli val_loss nie maleje przez 20 epok, przerwij trening
es = EarlyStopping(
    monitor='val_loss',
    verbose=1,
    patience=20)         # Czekaj 20 epok

# ROZPOCZĘCIE TRENINGU
print("\n=== ROZPOCZĘCIE TRENINGU ===")
print(f"Epoki: {epochs}")
print(f"Batch size: {batch_size}")
print(f"Kroki na epokę: {steps_per_epoch}")
print(f"Próbki treningowe: {len(x_train)}")
print(f"Próbki walidacyjne: {len(x_val)}")
print("\nTrening może zająć kilka godzin. Poczekaj cierpliwie...\n")

history = model.fit(
    train_generator,              # Generator danych treningowych z augmentacją
    steps_per_epoch=steps_per_epoch,
    epochs=epochs,
    validation_data=val_generator,  # Dane walidacyjne
    callbacks=[lr, es, cp])         # Callbacki

print("\n=== TRENING ZAKOŃCZONY ===")

In [ ]:
# Wizualizacja przebiegu treningu
# Sprawdź czy:
# - Loss maleje (dla train i val)
# - Accuracy rośnie (dla train i val)
# - Czy jest overfitting? (duża różnica train vs val)
plot_metrics(history)

In [ ]:
# EWALUACJA NA ZBIORZE TESTOWYM

# Wczytaj najlepsze wagi (zapisane podczas treningu)
model.load_weights(model.checkpoint_path)

# Oceń dokładność dla każdej emocji osobno
print("\n=== WYNIKI NA ZBIORZE TESTOWYM ===")
print("(Model NIE widział tych danych podczas treningu)\n")
evaluate_class(model, x_test, y_test)

In [ ]:
# Kopiowanie wytrenowanego modelu do Google Drive
# UWAGA: Dostosuj ścieżkę do swojej struktury folderów!
# Dzięki temu model będzie dostępny w innych notebookach
%cp /content/project/run/vggnet.h5 /content/gdrive/MyDrive/loopQ/project/saved_models

# **Gratulacje!**

Właśnie wytrenowałeś model rozpoznawania emocji!

**Co dalej?**
1. Przejdź do notebooka `inference.ipynb` aby przetestować model
2. Spróbuj dostroić hiperparametry (learning rate, liczba warstw, itp.)
3. Eksperymentuj z różnymi architekturami

**Pytania do przemyślenia:**
- Która emocja była najtrudniejsza do nauczenia? Dlaczego?
- Czy augmentacja danych pomogła?
- Jak można poprawić wyniki modelu?